#  Notebook 2 — Faster R-CNN Training (ResNet-50 + FPN)

**Model:** `fasterrcnn_resnet50_fpn_v2` — pretrained on COCO (80 classes)


In [ ]:
# ============================================================
#  CONFIG —
# ============================================================
import os

DATASET_BASE    = "dataset"       
CHECKPOINT_DIR  = "checkpoints"  
MODEL_SAVE_PATH = "saved_model"  

# --- Subset (None = full dataset) ---
TRAIN_SUBSET_SIZE = 5000  
VAL_SUBSET_SIZE   = 500    

# --- Training ---
BATCH_SIZE   = 4 
NUM_EPOCHS   = 20     
LR           = 0.005  
LR_MOMENTUM  = 0.9    
WEIGHT_DECAY = 0.0005 
LR_STEP_SIZE = 7
LR_GAMMA     = 0.1
IMG_SIZE     = 800    
NUM_WORKERS  = 0  

# --- Class imbalance ---
USE_WEIGHTED_SAMPLING = True 

CLASSES = [
    "traffic light", "traffic sign", "car", "person", "bus",
    "truck", "rider", "bike", "motor", "train", "banner", "tuktuk"
]

SPLITS = {
    "train": os.path.join(DATASET_BASE, "train", "train"),
    "val"  : os.path.join(DATASET_BASE, "val",   "val"),
    "test" : os.path.join(DATASET_BASE, "test",  "test"),
}

print(" Config ready!")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Epochs     : {NUM_EPOCHS}")
print(f"   Train imgs : {TRAIN_SUBSET_SIZE}")
print(f"   LR         : {LR}")

## Cell 1 — Environment Check

In [ ]:
import torch
import torchvision

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"Device       : {DEVICE}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

# Time estimate
spe = (TRAIN_SUBSET_SIZE or 48329) // BATCH_SIZE
mins = spe * 0.15 / 60  # ~150ms/step on RTX 4060
print(f"\n  Estimated training time:")
print(f"   Steps/epoch : {spe:,}")
print(f"   ~{mins:.0f} min/epoch × {NUM_EPOCHS} = ~{mins*NUM_EPOCHS/60:.1f} hrs")

## Cell 2 — Dataset Class

In [ ]:
import os, json, random
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import torchvision.transforms as T

random.seed(42)
torch.manual_seed(42)


class EgyptDetectionDataset(Dataset):
    """
    Faster R-CNN expects:
      - image: FloatTensor [C, H, W] normalized to [0,1]
      - target: dict with keys 'boxes' (FloatTensor [N,4]), 'labels' (Int64Tensor [N])
    """
    def __init__(self, split_path, img_size=800, transforms=None):
        self.img_dir  = os.path.join(split_path, "images")
        self.ann_file = os.path.join(split_path, "annotations.json")
        self.img_size = img_size
        self.transforms = transforms

        with open(self.ann_file) as f:
            coco = json.load(f)

        # Build image id → annotations mapping
        self.images = coco["images"]
        self.img_id_to_anns = {}
        for ann in coco["annotations"]:
            iid = ann["image_id"]
            self.img_id_to_anns.setdefault(iid, []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.img_dir, img_info["file_name"])
        image    = Image.open(img_path).convert("RGB")

        # Resize
        orig_w, orig_h = image.size
        image = image.resize((self.img_size, self.img_size))
        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h

        # To tensor
        image = T.ToTensor()(image)  # [C,H,W] float32 in [0,1]

        # Build target
        anns    = self.img_id_to_anns.get(img_info["id"], [])
        boxes   = []
        labels  = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            x1 = x * scale_x;  y1 = y * scale_y
            x2 = (x+w) * scale_x; y2 = (y+h) * scale_y
            # Clamp to image bounds
            x1 = max(0, min(x1, self.img_size))
            y1 = max(0, min(y1, self.img_size))
            x2 = max(0, min(x2, self.img_size))
            y2 = max(0, min(y2, self.img_size))
            if x2 > x1 and y2 > y1: 
                boxes.append([x1, y1, x2, y2])
                labels.append(ann["category_id"] + 1)  # +1: background=0 in Faster R-CNN

        if boxes:
            boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels,
                  "image_id": torch.tensor([idx])}
        return image, target

    def get_class_counts(self):
        """Return annotation count per class — used for weighted sampling."""
        from collections import Counter
        counts = Counter()
        for anns in self.img_id_to_anns.values():
            for ann in anns:
                counts[ann["category_id"]] += 1
        return counts


def collate_fn(batch):
    return tuple(zip(*batch))


def make_weighted_sampler(dataset, n_classes):
    """
    WeightedRandomSampler: images containing rare classes are sampled more.
    Directly addresses car:bike = 25:1 imbalance.
    """
    class_counts = dataset.get_class_counts()
    total = sum(class_counts.values())
    class_weights = {cls: total / max(cnt, 1) for cls, cnt in class_counts.items()}

    sample_weights = []
    for img_info in dataset.images:
        anns = dataset.img_id_to_anns.get(img_info["id"], [])
        if anns:
            # Weight = max class weight in this image (rare class images get high weight)
            w = max(class_weights.get(a["category_id"], 1.0) for a in anns)
        else:
            w = 1.0
        sample_weights.append(w)

    return WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True
    )


# Load full datasets
print("Loading datasets...")
TRAIN_DATASET = EgyptDetectionDataset(SPLITS["train"], img_size=IMG_SIZE)
VAL_DATASET   = EgyptDetectionDataset(SPLITS["val"],   img_size=IMG_SIZE)
TEST_DATASET  = EgyptDetectionDataset(SPLITS["test"],  img_size=IMG_SIZE)

# Subset
def make_subset(ds, size):
    if size is None or size >= len(ds): return ds
    return Subset(ds, random.sample(range(len(ds)), size))

TRAIN_SUBSET = make_subset(TRAIN_DATASET, TRAIN_SUBSET_SIZE)
VAL_SUBSET   = make_subset(VAL_DATASET,   VAL_SUBSET_SIZE)

# Weighted sampler for class imbalance
base_ds = TRAIN_DATASET if not isinstance(TRAIN_SUBSET, Subset) else TRAIN_DATASET
sampler = make_weighted_sampler(base_ds, len(CLASSES)) if USE_WEIGHTED_SAMPLING else None

TRAIN_LOADER = DataLoader(
    TRAIN_SUBSET, batch_size=BATCH_SIZE,
    sampler=sampler if sampler else None,
    shuffle=(sampler is None),
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True
)
VAL_LOADER = DataLoader(
    VAL_SUBSET, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True
)
TEST_LOADER = DataLoader(
    TEST_DATASET, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\n{'Split':<8} {'Total':>8} {'Used':>8} {'Batches':>8}")
print("-" * 36)
print(f"{'Train':<8} {len(TRAIN_DATASET):>8,} {len(TRAIN_SUBSET):>8,} {len(TRAIN_LOADER):>8,}")
print(f"{'Val':<8} {len(VAL_DATASET):>8,} {len(VAL_SUBSET):>8,} {len(VAL_LOADER):>8,}")
print(f"{'Test':<8} {len(TEST_DATASET):>8,} {len(TEST_DATASET):>8,} {len(TEST_LOADER):>8,}")
print(f"\nWeighted sampling: {USE_WEIGHTED_SAMPLING}")
print(" Dataloaders ready!")

## Cell 3 — Load Pretrained Faster R-CNN

In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NUM_CLASSES = len(CLASSES) + 1  

# Load pretrained on COCO 
model = fasterrcnn_resnet50_fpn_v2(
    weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
)

# This is transfer learning — keeps pretrained backbone, replaces head
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Faster R-CNN (ResNet-50 + FPN) loaded")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")
print(f"   Classes          : {NUM_CLASSES} (including background)")
print(f"   Device           : {DEVICE}")

## Cell 4 — Freeze Backbone (speeds up training)
> Freeze backbone layers so only the detection head is trained.
> Useful for small subsets. Set `FREEZE_BACKBONE=False` for full fine-tuning.

In [ ]:
FREEZE_BACKBONE = True 

if FREEZE_BACKBONE:
    for name, param in model.named_parameters():
        if 'backbone' in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f" Backbone frozen — trainable params: {trainable:,}")
else:
    print(" Full fine-tuning — all layers trainable")

## Cell 5 — Optimizer & Scheduler

In [ ]:
import torch

params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=LR_MOMENTUM, weight_decay=WEIGHT_DECAY)

# StepLR: reduce LR by gamma every step_size epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA
)

print(f"   Optimizer: SGD (lr={LR}, momentum={LR_MOMENTUM})")
print(f"   LR scheduler: StepLR (step={LR_STEP_SIZE}, gamma={LR_GAMMA})")

## Cell 6 — Training Loop


In [ ]:
import os, time
import torch
import matplotlib.pyplot as plt

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_losses = []
val_losses   = []
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    epoch_loss = 0
    t0 = time.time()

    for i, (images, targets) in enumerate(TRAIN_LOADER):
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ⚙️ TUNE: gradient clip
        optimizer.step()

        epoch_loss += losses.item()

        if (i+1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} | Step {i+1}/{len(TRAIN_LOADER)} "
                  f"| Loss: {losses.item():.4f}", end='\r')

    avg_train_loss = epoch_loss / len(TRAIN_LOADER)
    train_losses.append(avg_train_loss)
    lr_scheduler.step()

    # ── VALIDATE ───────────────────────────────────────────
    model.train()  
    val_loss = 0
    with torch.no_grad():
        for images, targets in VAL_LOADER:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            val_loss += sum(loss for loss in loss_dict.values()).item()
    avg_val_loss = val_loss / len(VAL_LOADER)
    val_losses.append(avg_val_loss)

    elapsed = (time.time() - t0) / 60
    print(f"\n Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"train={avg_train_loss:.4f} | val={avg_val_loss:.4f} | "
          f"lr={optimizer.param_groups[0]['lr']:.6f} | {elapsed:.1f} min")

    # Save checkpoint every epoch
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1:03d}_val{avg_val_loss:.3f}.pth")
    torch.save({
        'epoch'      : epoch,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'train_loss' : avg_train_loss,
        'val_loss'   : avg_val_loss,
    }, ckpt_path)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
        print(f"    New best model saved! (val_loss={best_val_loss:.4f})")

# Plot loss curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses,   label='Val Loss',   linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Faster R-CNN Training Loss')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n Training complete! Best val loss: {best_val_loss:.4f}")
print(f"   Best model: {CHECKPOINT_DIR}/best_model.pth")

## Cell 7 — Resume Training

In [ ]:
RESUME_PATH  = os.path.join(CHECKPOINT_DIR, "best_model.pth")  # ⚙️ TUNE
RESUME_EPOCH = 0  # auto-detected from checkpoint

if not os.path.exists(RESUME_PATH):
    print(f" Checkpoint not found: {RESUME_PATH}")
else:
    ckpt = torch.load(RESUME_PATH, map_location=DEVICE)
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optim_state'])
        RESUME_EPOCH = ckpt.get('epoch', 0) + 1
        print(f" Resumed from epoch {RESUME_EPOCH}")
    else:
        model.load_state_dict(ckpt)
        print(f" Loaded weights from: {RESUME_PATH}")
    print("   Re-run Cell 6 to continue training")

## Cell 8 — Save Final Model

In [ ]:
import os, torch
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# Load best weights and save
best_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f" Loaded best weights from: {best_path}")

save_path = os.path.join(MODEL_SAVE_PATH, 'fasterrcnn_egypt.pth')
torch.save(model.state_dict(), save_path)
size_mb = os.path.getsize(save_path) / 1024**2
print(f" Saved: {save_path} ({size_mb:.1f} MB)")

## Cell 9 — List Checkpoints

In [ ]:
ckpts = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth')])
print(f"Checkpoints in '{CHECKPOINT_DIR}/' ({len(ckpts)} files):")
for ckpt in ckpts:
    mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, ckpt)) / 1024**2
    print(f"   {ckpt:<50} {mb:.0f} MB")